# 06. 條件路由與分支

學習如何在 LangGraph 中實現複雜的條件路由邏輯。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 使用 `add_conditional_edges` 實現分支
- ✅ 建立基於規則的路由器
- ✅ 實現循環控制與終止條件
- ✅ 處理多條件組合路由

---

## 📊 條件路由架構

```
┌─────────────────────────────────────────────────────────┐
│                     條件路由模式                         │
├─────────────────────────────────────────────────────────┤
│                                                         │
│                    ┌─────────────┐                      │
│                    │   START     │                      │
│                    └──────┬──────┘                      │
│                           │                             │
│                    ┌──────▼──────┐                      │
│                    │   分類器    │                      │
│                    └──────┬──────┘                      │
│                           │                             │
│              ┌────────────┼────────────┐                │
│              │            │            │                │
│              ▼            ▼            ▼                │
│        ┌─────────┐  ┌─────────┐  ┌─────────┐           │
│        │ 路徑 A  │  │ 路徑 B  │  │ 路徑 C  │           │
│        └────┬────┘  └────┬────┘  └────┬────┘           │
│             │            │            │                 │
│             └────────────┴────────────┘                 │
│                          │                              │
│                   ┌──────▼──────┐                       │
│                   │     END     │                       │
│                   └─────────────┘                       │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 核心 API

```python
graph.add_conditional_edges(
    source="node_name",         # 來源節點
    path=router_function,        # 路由函數
    path_map={                   # 路由映射
        "route_a": "node_a",
        "route_b": "node_b",
    }
)
```

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

---

## 6.1 基本條件路由

### 場景：根據查詢類型分流處理

| 查詢類型 | 關鍵字 | 處理節點 |
|---------|--------|----------|
| 天氣 | 天氣, weather | weather_handler |
| 數學 | 計算, 加, 乘 | math_handler |
| 一般 | 其他 | general_handler |

In [2]:
class RouterState(TypedDict):
    """路由狀態"""
    query: str        # 使用者查詢
    query_type: str   # 分類後的類型
    result: str       # 處理結果

def classify_query(state: RouterState) -> dict:
    """分類查詢類型（基於規則）"""
    query = state["query"].lower()
    
    if "天氣" in query or "weather" in query:
        query_type = "weather"
    elif "計算" in query or "加" in query or "乘" in query:
        query_type = "math"
    else:
        query_type = "general"
    
    print(f"  📋 分類: '{query[:20]}...' → {query_type}")
    return {"query_type": query_type}

def weather_handler(state: RouterState) -> dict:
    print("  🌤️ 天氣處理器執行")
    return {"result": f"🌤️ 處理天氣查詢: {state['query']}"}

def math_handler(state: RouterState) -> dict:
    print("  🔢 數學處理器執行")
    return {"result": f"🔢 處理數學計算: {state['query']}"}

def general_handler(state: RouterState) -> dict:
    print("  💬 一般處理器執行")
    return {"result": f"💬 處理一般問題: {state['query']}"}

print("✅ 處理器定義完成")

✅ 處理器定義完成


In [3]:
def route_query(state: RouterState) -> Literal["weather", "math", "general"]:
    """路由函數：返回下一個節點名稱
    
    Returns:
        str: 必須是 path_map 中定義的 key
    """
    return state["query_type"]

# 建構圖
graph = StateGraph(RouterState)

# 添加節點
graph.add_node("classify", classify_query)
graph.add_node("weather", weather_handler)
graph.add_node("math", math_handler)
graph.add_node("general", general_handler)

# 添加邊
graph.add_edge(START, "classify")

# 🔑 關鍵：條件邊
graph.add_conditional_edges(
    source="classify",     # 從這個節點出發
    path=route_query,       # 使用這個函數決定路徑
    path_map={              # 路徑映射
        "weather": "weather",
        "math": "math",
        "general": "general"
    }
)

# 所有處理器都連到 END
graph.add_edge("weather", END)
graph.add_edge("math", END)
graph.add_edge("general", END)

app = graph.compile()
print("✅ 條件路由圖建構完成！")

✅ 條件路由圖建構完成！


In [4]:
# 視覺化
print("📊 圖結構:")
print(app.get_graph().draw_mermaid())

📊 圖結構:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	weather(weather)
	math(math)
	general(general)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> general;
	classify -.-> math;
	classify -.-> weather;
	general --> __end__;
	math --> __end__;
	weather --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [5]:
# 測試不同類型的查詢
print("📊 測試條件路由：")
print("=" * 50)

queries = [
    "今天台北天氣如何？",
    "請幫我計算 15 加 27",
    "你是誰？"
]

for q in queries:
    print(f"\n查詢: {q}")
    print("-" * 30)
    result = app.invoke({"query": q, "query_type": "", "result": ""})
    print(f"結果: {result['result']}")

📊 測試條件路由：

查詢: 今天台北天氣如何？
------------------------------
  📋 分類: '今天台北天氣如何？...' → weather
  🌤️ 天氣處理器執行
結果: 🌤️ 處理天氣查詢: 今天台北天氣如何？

查詢: 請幫我計算 15 加 27
------------------------------
  📋 分類: '請幫我計算 15 加 27...' → math
  🔢 數學處理器執行
結果: 🔢 處理數學計算: 請幫我計算 15 加 27

查詢: 你是誰？
------------------------------
  📋 分類: '你是誰？...' → general
  💬 一般處理器執行
結果: 💬 處理一般問題: 你是誰？


---

## 6.2 循環控制

### 循環模式架構

```
┌─────────────────────────────────────────┐
│              循環控制模式               │
├─────────────────────────────────────────┤
│                                         │
│   START ──▶ process ──▶ 檢查條件        │
│                 ▲           │           │
│                 │      ┌────┴────┐      │
│                 │      │         │      │
│                 │   continue    end     │
│                 │      │         │      │
│                 └──────┘         ▼      │
│                                END      │
│                                         │
└─────────────────────────────────────────┘
```

**重點**：`continue` 路徑回到 `process`，形成循環

In [6]:
class LoopState(TypedDict):
    """循環狀態"""
    count: int                                      # 當前計數
    max_count: int                                  # 最大次數
    values: Annotated[list[int], lambda x, y: x + y]  # 累加結果

def process_step(state: LoopState) -> dict:
    """處理一個步驟"""
    new_count = state["count"] + 1
    new_value = new_count * 2
    print(f"  步驟 {new_count}: 產生值 {new_value}")
    return {
        "count": new_count,
        "values": [new_value]
    }

def should_continue(state: LoopState) -> Literal["continue", "end"]:
    """檢查是否應該繼續
    
    這是循環控制的關鍵！
    """
    if state["count"] >= state["max_count"]:
        print(f"  ⏹️ 達到最大次數 {state['max_count']}，停止")
        return "end"
    return "continue"

# 建構循環圖
loop_graph = StateGraph(LoopState)
loop_graph.add_node("process", process_step)

loop_graph.add_edge(START, "process")
loop_graph.add_conditional_edges("process", should_continue, {
    "continue": "process",  # 🔄 回到自己形成循環
    "end": END
})

loop_app = loop_graph.compile()
print("✅ 循環圖建構完成")

✅ 循環圖建構完成


In [7]:
print("📊 測試循環控制：")
print("=" * 50)

result = loop_app.invoke({"count": 0, "max_count": 5, "values": []})

print("=" * 50)
print(f"\n📋 最終結果: {result['values']}")
print(f"   總共執行 {result['count']} 次")

📊 測試循環控制：
  步驟 1: 產生值 2
  步驟 2: 產生值 4
  步驟 3: 產生值 6
  步驟 4: 產生值 8
  步驟 5: 產生值 10
  ⏹️ 達到最大次數 5，停止

📋 最終結果: [2, 4, 6, 8, 10]
   總共執行 5 次


---

## 6.3 多條件組合

### 場景：根據用戶等級 + 請求類型決定處理方式

| 用戶類型 | 請求類型 | 處理方式 |
|---------|---------|----------|
| enterprise | 任意 | 優先處理 |
| premium | 任意 | 完整服務 |
| free | simple | 基本處理 |
| free | complex | 提示升級 |

In [8]:
class MultiConditionState(TypedDict):
    """多條件狀態"""
    user_type: str    # "free" | "premium" | "enterprise"
    request_type: str # "simple" | "complex"
    result: str

def analyze_request(state: MultiConditionState) -> dict:
    print(f"  📋 分析: user={state['user_type']}, request={state['request_type']}")
    return state

def free_simple(state): return {"result": "✅ 免費用戶 - 基本處理"}
def free_complex(state): return {"result": "⚠️ 免費用戶 - 複雜請求需升級"}
def premium_handler(state): return {"result": "🌟 Premium 用戶 - 完整服務"}
def enterprise_handler(state): return {"result": "👑 Enterprise 用戶 - 優先處理"}

def multi_router(state: MultiConditionState) -> str:
    """多條件路由邏輯
    
    優先級：enterprise > premium > free
    """
    user = state["user_type"]
    req = state["request_type"]
    
    if user == "enterprise":
        return "enterprise"
    elif user == "premium":
        return "premium"
    elif req == "simple":
        return "free_simple"
    else:
        return "free_complex"

# 建構圖
multi_graph = StateGraph(MultiConditionState)
multi_graph.add_node("analyze", analyze_request)
multi_graph.add_node("free_simple", free_simple)
multi_graph.add_node("free_complex", free_complex)
multi_graph.add_node("premium", premium_handler)
multi_graph.add_node("enterprise", enterprise_handler)

multi_graph.add_edge(START, "analyze")
multi_graph.add_conditional_edges("analyze", multi_router)
for node in ["free_simple", "free_complex", "premium", "enterprise"]:
    multi_graph.add_edge(node, END)

multi_app = multi_graph.compile()
print("✅ 多條件路由圖建構完成")

✅ 多條件路由圖建構完成


In [9]:
print("📊 測試多條件路由：")
print("=" * 50)

test_cases = [
    {"user_type": "free", "request_type": "simple"},
    {"user_type": "free", "request_type": "complex"},
    {"user_type": "premium", "request_type": "complex"},
    {"user_type": "enterprise", "request_type": "simple"},
]

for case in test_cases:
    result = multi_app.invoke({**case, "result": ""})
    print(f"  {case['user_type']:12} + {case['request_type']:8} → {result['result']}")

📊 測試多條件路由：
  📋 分析: user=free, request=simple
  free         + simple   → ✅ 免費用戶 - 基本處理
  📋 分析: user=free, request=complex
  free         + complex  → ⚠️ 免費用戶 - 複雜請求需升級
  📋 分析: user=premium, request=complex
  premium      + complex  → 🌟 Premium 用戶 - 完整服務
  📋 分析: user=enterprise, request=simple
  enterprise   + simple   → 👑 Enterprise 用戶 - 優先處理


---

## 💡 重點回顧

### add_conditional_edges 參數

```python
graph.add_conditional_edges(
    source,      # 來源節點名稱
    path,        # 路由函數：接收 state，返回路徑 key
    path_map     # 可選：{"key": "node_name"}
)
```

### 路由函數規則

| 規則 | 說明 |
|------|------|
| 輸入 | 完整的 State |
| 輸出 | 字串（路徑 key 或節點名） |
| 類型標註 | 使用 `Literal["a", "b"]` |
| 特殊值 | 可返回 `END` 直接結束 |

### 循環控制要點

1. **終止條件**：必須有明確的終止邏輯
2. **計數器**：防止無限循環
3. **狀態累加**：用 Reducer 累積結果

---

## 📝 練習題

1. **時間路由**：根據當前時間決定問候語（早安/午安/晚安）
2. **重試邏輯**：實作失敗時自動重試（最多 3 次）
3. **訊息長度路由**：短訊息走快速處理，長訊息走詳細處理
4. **優先級隊列**：根據任務優先級分配到不同處理器

---

下一步：[07. 記憶體與持久化](07_memory.ipynb)